1. Imports & config

In [22]:
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

TICKERS = [
    "TCS.NS", "INFY.NS", "WIPRO.NS", "HCLTECH.NS", "TECHM.NS",
    "LTM.NS", "MPHASIS.NS", "PERSISTENT.NS", "COFORGE.NS", "LTTS.NS",
]

LOOKBACK_YEARS = 5
WINDOW_SIZE = 30
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
OUTPUT_DIR = "data/technical"

FEATURE_COLUMNS = [
    "open", "high", "low", "close", "volume",
    "sma_10", "sma_20", "sma_50",
    "ema_12", "ema_26",
    "macd", "macd_signal", "macd_hist",
    "rsi_14",
    "bb_upper", "bb_lower", "bb_width",
    "atr_14",
    "obv",
    "adi",
    "stoch_k", "stoch_d",
    "williams_r",
    "bull_power", "bear_power",
    "daily_return", "log_return",
]

2. Fetch raw OHLCV

In [23]:
def fetch_raw(ticker: str, years: int = LOOKBACK_YEARS) -> pd.DataFrame:
    df = yf.download(
        ticker,
        period=f"{years}y",
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    if df.empty:
        raise ValueError(f"No data returned for {ticker}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [c.lower() for c in df.columns]
    df.index.name = "date"
    return df

3. Clean raw data

In [24]:
def clean_raw(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df[~df.index.duplicated(keep="first")]
    df = df.sort_index()

    bad_rows = (df["volume"] <= 0) | (df["close"] <= 0)
    df = df[~bad_rows]

    df[["open", "high", "low", "close", "volume"]] = (
        df[["open", "high", "low", "close", "volume"]].ffill(limit=3)
    )

    df = df.dropna(subset=["open", "high", "low", "close", "volume"])
    return df

4. Add technical indicators (now includes ADI, Williams %R, Bull/Bear Power)

In [20]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    close, high, low, volume = df["close"], df["high"], df["low"], df["volume"]

    # --- Trend: SMA / EMA ---
    df["sma_10"] = close.rolling(10).mean()
    df["sma_20"] = close.rolling(20).mean()
    df["sma_50"] = close.rolling(50).mean()
    df["ema_12"] = close.ewm(span=12, adjust=False).mean()
    df["ema_26"] = close.ewm(span=26, adjust=False).mean()

    # --- MACD ---
    df["macd"] = df["ema_12"] - df["ema_26"]
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd"] - df["macd_signal"]

    # --- RSI (14) ---
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / 14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))
    df["rsi_14"] = df["rsi_14"].fillna(50)

    # --- Bollinger Bands (20, 2std) ---
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    df["bb_upper"] = bb_mid + 2 * bb_std
    df["bb_lower"] = bb_mid - 2 * bb_std
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / bb_mid

    # --- ATR (14) ---
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)
    df["atr_14"] = tr.ewm(alpha=1 / 14, adjust=False).mean()

    # --- OBV ---
    direction = np.sign(close.diff()).fillna(0)
    df["obv"] = (direction * volume).cumsum()

    # --- ADI (Accumulation/Distribution Index) ---
    clv = ((close - low) - (high - close)) / (high - low).replace(0, np.nan)
    clv = clv.fillna(0)  # high == low -> no clear signal, treated as neutral (0)
    df["adi"] = (clv * volume).cumsum()

    # --- Stochastic Oscillator (14, 3) ---
    low_14 = low.rolling(14).min()
    high_14 = high.rolling(14).max()
    df["stoch_k"] = 100 * (close - low_14) / (high_14 - low_14).replace(0, np.nan)
    df["stoch_d"] = df["stoch_k"].rolling(3).mean()

    # --- Williams %R (14) ---
    df["williams_r"] = -100 * (high_14 - close) / (high_14 - low_14).replace(0, np.nan)

    # --- Bull/Bear Power (Elder Ray Index, EMA-13 based) ---
    ema_13 = close.ewm(span=13, adjust=False).mean()
    df["bull_power"] = high - ema_13
    df["bear_power"] = low - ema_13

    # --- Returns ---
    df["daily_return"] = close.pct_change()
    df["log_return"] = np.log(close / close.shift(1))

    return df


def drop_warmup_nans(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(subset=FEATURE_COLUMNS).reset_index(drop=False)

In [21]:
5. Split + scale

SyntaxError: invalid syntax (2877582933.py, line 1)

In [ ]:
def chronological_split(df: pd.DataFrame):
    n = len(df)
    train_end = int(n * TRAIN_FRAC)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))
    return df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]


def scale_splits(train_df, val_df, test_df):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(train_df[FEATURE_COLUMNS])

    train_scaled = scaler.transform(train_df[FEATURE_COLUMNS])
    val_scaled = scaler.transform(val_df[FEATURE_COLUMNS])
    test_scaled = scaler.transform(test_df[FEATURE_COLUMNS])

    return (train_scaled, val_scaled, test_scaled), scaler

6. Windowing

In [ ]:
def make_sequences(arr: np.ndarray, window: int = WINDOW_SIZE):
    close_idx = FEATURE_COLUMNS.index("close")
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i + window])
        y.append(arr[i + window, close_idx])
    return np.array(X), np.array(y)

7. Orchestration

In [25]:
def run_pipeline(tickers=TICKERS, out_dir=OUTPUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    manifest = {}

    for ticker in tickers:
        try:
            raw = fetch_raw(ticker)
            clean = clean_raw(raw)
            featured = add_technical_indicators(clean)
            featured = drop_warmup_nans(featured)

            if len(featured) < WINDOW_SIZE + 50:
                print(f"[skip] {ticker}: not enough rows after cleaning ({len(featured)})")
                continue

            train_df, val_df, test_df = chronological_split(featured)
            (train_s, val_s, test_s), scaler = scale_splits(train_df, val_df, test_df)

            X_train, y_train = make_sequences(train_s)
            X_val, y_val = make_sequences(val_s)
            X_test, y_test = make_sequences(test_s)

            stock_dir = os.path.join(out_dir, ticker.replace(".", "_"))
            os.makedirs(stock_dir, exist_ok=True)

            np.save(os.path.join(stock_dir, "technical_X_train.npy"), X_train)
            np.save(os.path.join(stock_dir, "technical_y_train.npy"), y_train)
            np.save(os.path.join(stock_dir, "technical_X_val.npy"), X_val)
            np.save(os.path.join(stock_dir, "technical_y_val.npy"), y_val)
            np.save(os.path.join(stock_dir, "technical_X_test.npy"), X_test)
            np.save(os.path.join(stock_dir, "technical_y_test.npy"), y_test)

            featured.to_csv(os.path.join(stock_dir, "technical_featured_unscaled.csv"), index=False)

            manifest[ticker] = {
                "rows_after_cleaning": len(featured),
                "date_range": [str(featured["date"].min()), str(featured["date"].max())],
                "train_shape": list(X_train.shape),
                "val_shape": list(X_val.shape),
                "test_shape": list(X_test.shape),
                "feature_columns": FEATURE_COLUMNS,
            }
            print(f"[ok] {ticker}: train={X_train.shape} val={X_val.shape} test={X_test.shape}")

        except Exception as e:
            print(f"[error] {ticker}: {e}")

    with open(os.path.join(out_dir, "technical_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nDone. Manifest written to {out_dir}/technical_manifest.json")
    return manifest


if __name__ == "__main__":
    run_pipeline()

[ok] TCS.NS: train=(800, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] INFY.NS: train=(800, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] WIPRO.NS: train=(800, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] HCLTECH.NS: train=(800, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] TECHM.NS: train=(799, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] LTM.NS: train=(798, 30, 27) val=(147, 30, 27) test=(148, 30, 27)
[ok] MPHASIS.NS: train=(799, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] PERSISTENT.NS: train=(799, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] COFORGE.NS: train=(799, 30, 27) val=(148, 30, 27) test=(148, 30, 27)
[ok] LTTS.NS: train=(799, 30, 27) val=(148, 30, 27) test=(148, 30, 27)

Done. Manifest written to data/technical/technical_manifest.json
